# Embedding Space Visualisation

This notebook investigates how embeddings extracted from the model are organised within the embedding space.

The embedding space is analysed using several dimensionality reduction techniques, including:

- PCA
- t-SNE
- UMAP

Analysis is performed at both the image level, using CLS embeddings, and the local level, using patch embeddings. This allows investigation of how entire images relate to one another within the embedding space, as well as how individual patches relate to other patches within the same image.

Scoring metrics will be used to analyse cluster quality and separation, including:

- Silhouette Score
- Intra-Cluster Distance
- Inter-Cluster Distance

The objective of this notebook is to understand the structure of the embedding space and identify characteristics that can help with anomaly detection. These findings will be used to explain the performance of the baseline model and provide a reference for future experiments involving fine-tuning and alternative embedding representations.

In [1]:
import os 

import torch
import pandas as pd 
import numpy as np
import cv2
import sqlite3
import pickle

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import colorsys

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, IMAGES, CACHES, RESULTS, DB_PATH

EMBED_PATH = EMBEDS_DIR / "dino/pretrained"
EMBED_NAME = EMBED_PATH.stem

IMAGE_DIR = IMAGES / "emb_visuals" / EMBED_NAME
CACHE_DIR = CACHES / EMBED_NAME
RESULTS_DIR = RESULTS / EMBED_NAME

In [2]:
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)
patches = torch.load(EMBED_PATH/"patches.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()
types_per_cat = pd.read_sql_query("""
    SELECT category, COUNT(DISTINCT type) AS num_types
    FROM meta
    GROUP BY category
""", conn)
max_types = types_per_cat["num_types"].max()

conn.close()

In [5]:
def load_or_compute(path, fn):
    if path.exists():
        print("Loading ", path)
        return np.load(path)
    
    print("Computing ", path)
    result = fn()
    np.save(path, result)
    return result

In [6]:
#fix bug here it reloads the old dim_reduction compared by make time maybe?
pca = PCA(n_components=2)
tsne = TSNE(n_components=2)
umap = UMAP(n_components=2, random_state=42)

train_mask = meta["split"] == "train"
train_meta = meta[train_mask]
test_meta = meta[~train_mask]

pca_2d = load_or_compute(CACHE_DIR / "cls_pca.npy", lambda: pca.fit_transform(cls_tokens))
tsne_2d = load_or_compute(CACHE_DIR / "cls_tsne.npy", lambda: tsne.fit_transform(cls_tokens))
umap_2d = load_or_compute(CACHE_DIR/ "cls_umap.npy", lambda: umap.fit_transform(cls_tokens))


Loading  /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/data/cache/pretrained/cls_pca.npy
Loading  /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/data/cache/pretrained/cls_tsne.npy
Loading  /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/data/cache/pretrained/cls_umap.npy


In [7]:
pca_train = pca_2d[train_mask]
tsne_train = tsne_2d[train_mask]
umap_train = umap_2d[train_mask]

cls_fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("PCA", "t-SNE", "UMAP")
)

colors = px.colors.qualitative.Dark24

category_colors = {cat: colors[i % len(colors)] for i, cat in enumerate(categories)}
plot_meta = [("PCA", pca_train, 1), ("t-SNE", tsne_train, 2), ("UMAP", umap_train, 3)]

for idx, (method_name, coords, col) in enumerate(plot_meta):
    for category in categories:
        mask = train_meta["category"] == category

        cls_fig.add_trace(
            go.Scattergl(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=category,
                legendgroup=category,
                showlegend=(method_name == "PCA"),
                customdata=np.stack([
                    train_meta.loc[mask, "category"],
                    train_meta.loc[mask, "path"]
                ], axis=1),
                hovertemplate=(
                    "Category: %{customdata[0]}" +
                    "<br>Path: %{customdata[1]}" +
                    "<extra></extra>"
                ),
                marker=dict(
                    size=4,
                    color=category_colors[category]
                    )
            ),
            row=1,
            col=col
        )

cls_fig.update_xaxes(title_text="PC1", row=1, col=1)
cls_fig.update_yaxes(title_text="PC2", row=1, col=1)

cls_fig.update_xaxes(title_text="t-SNE1", row=1, col=2)
cls_fig.update_yaxes(title_text="t-SNE2", row=1, col=2)

cls_fig.update_xaxes(title_text="UMAP1", row=1, col=3)
cls_fig.update_yaxes(title_text="UMAP2", row=1, col=3)

cls_fig.update_layout(
    title="CLS Embedding Projections",
    width=1600, height=600,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,
        xanchor="center",
        x=0.5
    )
)

cls_fig.show()
cls_fig.write_image(IMAGE_DIR / "cls_embedding_comparison.svg")

Within the PCA projection, we can observe how the embedding space is organised along the two principal components that capture the greatest variance in the data. Several categories form distinct clusters that reflect similarities in their visual appearance. For example, relatively flat and homogeneous objects such as tile, leather, and wood are positioned closer together, while categories such as screw occupy a more distant region of the embedding space due to their more complex geometric structure and visual texture.

##

In the t-SNE projection, we can observe how compact or diffuse each category is within the embedding space. Most categories form relatively tight clusters, indicating that samples from the same category are represented consistently by the model. Notable exceptions include grid and screw, which exhibit more diffuse and fragmented structures. The screw category is particularly interesting, as it also achieved some of the lowest anomaly detection performance in the previous analysis. Rather than forming a single compact cluster, the embeddings are distributed in a figure-eight like structure, suggesting that normal and defective samples may occupy multiple overlapping regions of the embedding space. This reduced compactness may contribute to the lower AUROC scores observed for this category, as anomalous samples become more difficult to distinguish from normal samples using anomaly scoring methods.

##

Finally, the UMAP projection highlights how distinct the local neighbourhoods are within the embedding space and how compact these neighbourhoods remain. The observations regarding cluster compactness are similar to those seen in the t-SNE projection, with grid and screw again emerging as the most diffuse categories. Rather than forming a single coherent cluster, samples from these categories occupy multiple regions of the embedding space, with the screw category appearing to split into three distinct subregions. This provides further insight into the lower anomaly detection performance observed for screw in the previous analysis. Since normal samples are distributed across several disconnected regions rather than a single compact cluster, anomaly scoring methods may struggle to distinguish normal and defective samples consistently, resulting in reduced AUROC performance.

In [8]:

def hsv_to_hex(h, s=0.80, v=0.90):
    r, g, b = colorsys.hsv_to_rgb(h / 360, s, v)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

category_colours = {}

GOLDEN_ANGLE = 137.508

for i, category in enumerate(categories):
    normal_hue = (i * GOLDEN_ANGLE) % 360
    defect_hue = (normal_hue + 180) % 360

    category_colours[category] = {
        "normal": hsv_to_hex(normal_hue),
        "defect": hsv_to_hex(defect_hue)
    }

print(category_colours)

{'bottle': {'normal': '#e52d2d', 'defect': '#2de5e5'}, 'cable': {'normal': '#2de563', 'defect': '#e52daf'}, 'capsule': {'normal': '#992de5', 'defect': '#7ae52d'}, 'carpet': {'normal': '#e5ce2d', 'defect': '#2d44e5'}, 'grid': {'normal': '#2dc6e5', 'defect': '#e54c2d'}, 'hazelnut': {'normal': '#e52d91', 'defect': '#2de582'}, 'leather': {'normal': '#5be52d', 'defect': '#b72de5'}, 'metal_nut': {'normal': '#352de5', 'defect': '#dde52d'}, 'pill': {'normal': '#e56b2d', 'defect': '#2da8e5'}, 'screw': {'normal': '#2de5a0', 'defect': '#e52d72'}, 'tile': {'normal': '#d62de5', 'defect': '#3ce52d'}, 'toothbrush': {'normal': '#bee52d', 'defect': '#542de5'}, 'transistor': {'normal': '#2d89e5', 'defect': '#e5892d'}, 'wood': {'normal': '#e52d53', 'defect': '#2de5bf'}, 'zipper': {'normal': '#2de53d', 'defect': '#e52dd5'}}


In [9]:
pca_test = pca_2d[~train_mask]
tsne_test = tsne_2d[~train_mask]
umap_test = umap_2d[~train_mask]

def_fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("PCA", "t-SNE", "UMAP")
)

plot_meta = [("PCA", pca_test, 1), ("t-SNE", tsne_test, 2), ("UMAP", umap_test, 3)]
for idx, (method_name, coords, col) in enumerate(plot_meta):
    for category in categories:
        mask = test_meta["category"] == category
        good_mask = mask & (test_meta["type"] == "good")
        bad_mask = mask & (test_meta["type"] != "good")

        normal_colour = category_colours[category]["normal"]
        defect_colour = category_colours[category]["defect"]

        def_fig.add_trace(
            go.Scattergl(
                x=coords[good_mask, 0],
                y=coords[good_mask, 1],
                mode="markers",
                marker=dict(size=4, color=normal_colour, symbol="circle"),
                name="Normal",
                showlegend=(method_name == "PCA"),
                legendgroup=category,
                legendgrouptitle_text=category,
                customdata=np.stack([
                    test_meta.loc[good_mask, "category"],
                    test_meta.loc[good_mask, "type"],
                    test_meta.loc[good_mask, "path"]
                ], axis=1),
                hovertemplate=(
                    "Category: %{customdata[0]}" +
                    "<br>Type: %{customdata[1]}" +
                    "<br>Path: %{customdata[2]}" +
                    "<extra></extra>"
                )
            ),
            row=1,
            col=col
        )

        def_fig.add_trace(
            go.Scattergl(
                x=coords[bad_mask, 0],
                y=coords[bad_mask, 1],
                mode="markers",
                marker=dict(size=4, color=defect_colour, symbol="x"),
                name="Defective",
                showlegend=(method_name == "PCA"),
                legendgroup=category,
                legendgrouptitle_text=category,
                customdata=np.stack([
                    test_meta.loc[bad_mask, "category"],
                    test_meta.loc[bad_mask, "type"],
                    test_meta.loc[bad_mask, "path"]
                ], axis=1),
                hovertemplate=(
                    "Category: %{customdata[0]}" +
                    "<br>Type: %{customdata[1]}" +
                    "<br>Path: %{customdata[2]}" +
                    "<extra></extra>"
                )
            ),
            row=1,
            col=col
        )

def_fig.update_xaxes(title_text="PC1", row=1, col=1)
def_fig.update_yaxes(title_text="PC2", row=1, col=1)

def_fig.update_xaxes(title_text="t-SNE1", row=1, col=2)
def_fig.update_yaxes(title_text="t-SNE2", row=1, col=2)

def_fig.update_xaxes(title_text="UMAP1", row=1, col=3)
def_fig.update_yaxes(title_text="UMAP2", row=1, col=3)

def_fig.update_layout(
    title="CLS Embedding Projections(Normal vs Defective)",
    width=1600, height=600,
    legend=dict(
        itemsizing="constant",
        font=dict(size=8),
        itemwidth=45,
        orientation="h",
        yanchor="top",
        y=-0.1,
        xanchor="center",
        x=0.5,
    )
)
def_fig.show()
def_fig.write_image(IMAGE_DIR / "def_normal.svg")

For the PCA projection, there is unsurprisingly little separation between normal and defective samples. Since PCA preserves the directions of greatest global variance, the principal components are dominated by category-level differences rather than the subtle differences introduced by anomalies. As a result, normal and defective samples largely occupy the same regions of the embedding space.

##

Within the t-SNE projection, we can observe how defective samples are positioned relative to local neighbourhoods of normal samples. For many categories, particularly those containing flat and visually consistent objects, normal and defective samples form relatively distinct regions. However, the screw category remains an exception. Here, normal and defective samples occupy almost identical regions of the embedding space, with little evidence of a distinct anomaly cluster. This provides a possible explanation for the lower anomaly detection performance observed previously, as the embedding representation does not naturally separate normal and defective screw samples.

##

The UMAP projection exhibits similar behaviour to t-SNE. For many categories, defective samples tend to occupy regions adjacent to, but partially separated from, normal samples. However, in more challenging categories such as screw, normal and defective samples continue to share the same embedding regions. This suggests that the model struggles to learn discriminative representations for these anomalies, making them difficult to separate using distance-based anomaly scoring methods.

In [10]:
def compute_cat_metrics(cat_data):
    test_labels = cat_data["labels"]
    test_cls = cat_data["test"]
    good_mask = cat_data["mask"]
    
    sil = silhouette_score(test_cls, test_labels)

    train_centroid = cat_data["centroid"]
    test_good = test_cls[good_mask]
    test_def = test_cls[~good_mask]

    good_inter = np.mean(np.linalg.norm(test_good - train_centroid, axis=1))
    defect_inter = np.mean(np.linalg.norm(test_def - train_centroid, axis=1))

    sep_ratio = defect_inter / good_inter

    good_centroid = test_good.mean(axis=0)
    def_centroid = test_def.mean(axis=0)

    good_intra = np.mean(np.linalg.norm(test_good - good_centroid, axis=1))
    defect_intra = np.mean(np.linalg.norm(test_def - def_centroid, axis=1))

    db_index = davies_bouldin_score(test_cls, test_labels)
    ch_index = calinski_harabasz_score(test_cls, test_labels)

    return {
        "category": cat_data["name"],
        "silhouette": sil,
        "normal_inter": good_inter,
        "defect_inter": defect_inter,
        "separation": sep_ratio,
        "normal_intra": good_intra,
        "defect_intra": defect_intra,
        "davies_bouldin": db_index,
        "cal_har": ch_index
    }
    

In [11]:
cls_results_path = RESULTS_DIR / "cls_results.pkl"
cluster_results_path = RESULTS_DIR / "cluster_results.csv"
global_sil_path = RESULTS_DIR / "global_silhouette.csv"

if cluster_results_path.exists() and global_sil_path.exists():
    cluster_results = pd.read_csv(cluster_results_path)
    global_metrics = pd.read_csv(global_sil_path)

elif not cls_results_path.exists():
    print(f"Run {ROOT / 'notebooks/embedding_analysis/baseline.ipynb'} to create this file")

else:
    with open(cls_results_path, "rb") as f:
        cls_results = pickle.load(f)
    
    categories = meta.loc[train_mask, "category"].to_numpy()

    cls_sil = silhouette_score(cls_tokens[train_mask], categories)
    pca_sil = silhouette_score(pca_train, categories)
    tsne_sil = silhouette_score(tsne_train, categories)
    umap_sil = silhouette_score(umap_train, categories)

    global_metrics = pd.DataFrame([{
        "cls_silhouette": cls_sil,
        "pca_silhouette": pca_sil,
        "tsne_silhouette": tsne_sil,
        "umap_silhouette": umap_sil
    }])
    
    cat_data_list = []
    for cat in cls_results.keys():
        cat_meta = meta[
                (meta["category"] == cat) &
                (meta["split"] == "test")
            ]

        good_mask = cat_meta["type"].to_numpy() == "good"
        
        cat_data = {
            "name": cat,
            "mask": good_mask,
            "centroid": cls_results[cat]["centroid"],
            "test": cls_results[cat]["test"],
            "labels": cls_results[cat]["labels"]
        }

        cat_data_list.append(cat_data)

    rows = [
        compute_cat_metrics(cat_data)
        for cat_data in cat_data_list
    ]

    cluster_results = pd.DataFrame(rows)

    global_metrics.to_csv(global_sil_path, index=False)
    cluster_results.to_csv(cluster_results_path, index=False)

global_metrics

,cls_silhouette,pca_silhouette,tsne_silhouette,umap_silhouette
0,0.678885,0.736229,0.708068,0.878098


Based on the silhouette scores, t-SNE appears to produce a projection that most closely reflects the cluster structure present in the full CLS embedding space. However, this is just  an approximation rather than a definitive measure of projection quality, as silhouette scores only capture cluster compactness and separation and do not fully describe the geometry of the original high-dimensional space. The results therefore provide an indication of which projection best preserves the overall clustering behaviour of the model rather than a direct measure of faithfulness.

In [12]:
cluster_results

,category,silhouette,normal_inter,defect_inter,separation,normal_intra,defect_intra,davies_bouldin,cal_har
0,tile,0.051925,13.828683,30.462551,2.202853,13.194844,25.503618,2.151681,13.841351
1,zipper,0.054824,10.680494,17.839172,1.670257,10.037414,14.769583,2.255279,14.878088
2,toothbrush,0.010358,10.852008,19.439436,1.791322,10.421845,16.982060,2.601975,3.649014
3,hazelnut,0.035357,17.768284,23.515533,1.323456,16.975285,21.654016,3.417890,7.660800
4,wood,0.078673,21.327713,28.108612,1.317938,18.945887,23.288946,2.460468,8.035849
5,grid,0.075255,19.194645,22.477043,1.171006,18.273794,19.301464,2.893122,6.656239
6,cable,0.005631,17.824223,22.918423,1.285802,17.336468,22.108635,5.485366,4.156659
7,leather,0.161544,11.415674,26.463190,2.318145,9.138710,17.966806,1.505843,27.550618
8,metal_nut,0.008135,12.177718,15.678945,1.287511,11.461515,14.659174,3.217935,5.634620
9,screw,0.020685,16.529200,16.315859,0.987093,16.358189,15.930423,6.654463,2.655453


In [13]:
MINIMISE_METRICS = {
    "davies_bouldin",
    "normal_intra",
    "normal_inter"
}

MAXIMISE_METRICS = {
    "silhouette",
    "cal_har",
    "separation",
    "defect_inter",
    "defect_intra"
}

In [14]:
def column_stats(df, column):
    s = df[column]
    
    if column in MINIMISE_METRICS:
        best = df.loc[s.idxmin(), "category"]
        worst = df.loc[s.idxmax(), "category"]

    elif column in MAXIMISE_METRICS:
        best = df.loc[s.idxmax(), "category"]
        worst = df.loc[s.idxmin(), "category"]

    else:
        best = None
        worst = None
        
    return {
        "mean": round(s.mean(), 4),
        "median": round(s.median(), 4),
        "std": round(s.std(), 4),
        "min": round(s.min(), 4),
        "max": round(s.max(), 4),
        "worst_category": worst,
        "best_category": best
    }

In [15]:
numeric_cols = cluster_results.select_dtypes(include="number").columns

summary = pd.DataFrame({
    col: column_stats(cluster_results, col)
    for col in numeric_cols
}).T

summary.to_csv(RESULTS_DIR / "cls_embed_summary.csv")
summary

,mean,median,std,min,max,worst_category,best_category
silhouette,0.0401,0.0425,0.0588,-0.0722,0.1615,capsule,leather
normal_inter,13.5318,12.1777,4.0829,7.3152,21.3277,wood,bottle
defect_inter,20.5365,19.4394,5.0035,13.0549,30.4626,capsule,tile
separation,1.5956,1.3713,0.45,0.9871,2.4991,screw,bottle
normal_intra,12.6352,11.4615,3.9479,6.9367,18.9459,wood,bottle
defect_intra,17.7923,16.6142,3.7904,12.5704,25.5036,capsule,tile
davies_bouldin,3.2804,2.8931,1.4754,1.5058,6.6545,screw,leather
cal_har,8.8111,6.6562,6.7326,2.6555,27.5506,screw,leather


Silhouette Score has a very low mean, indicating that when all categories are considered simultaneously, the CLS embeddings do not form highly compact and well-separated clusters. This is expected given that many categories contain visually similar objects and that silhouette scores become more difficult to interpret when a large number of classes are present.

Defect Inter-Cluster Distance is consistently larger than Good Inter-Cluster Distance, suggesting that defective samples tend to lie further from the category centroid than normal samples. This supports the assumption that anomalies occupy more distant regions of the embedding space.

The mean Separation Ratio of approximately 1.60 indicates that defective samples are, on average, around 60% further from the category centroid than normal samples. This suggests that the pretrained CLS embeddings already contain a useful anomaly signal despite no task-specific training.

Good Intra-Cluster Distance is generally lower than Defect Intra-Cluster Distance, indicating that normal samples form more compact clusters while defective samples exhibit greater variability. This is consistent with anomalies introducing additional visual variation within a category.

Several metrics identify bottle as the best-performing category, while screw, capsule, frequently appear as the most challenging categories. This aligns with the earlier AUROC analysis, where screw consistently demonstrated poorer anomaly detection performance.

In [16]:
PLOT_DIR = IMAGE_DIR / "patch_plots"
HTML_CACHE = CACHE_DIR / "patch_plots"

patch_labels_path = CACHE_DIR / "patch_labels.pkl"
labels_exist = patch_labels_path.exists()

if labels_exist:
    with open(patch_labels_path, "rb") as f:
        all_labels = pickle.load(f)

patch_size = 14

def plot_patches():
    os.makedirs(PLOT_DIR, exist_ok=True)
    os.makedirs(HTML_CACHE, exist_ok=True)
    plot_figs = []

    for cat in categories:
        test_meta = meta[
                (meta["category"] == cat)
                & (meta["split"] == "test")
            ]
        
        test_samples = (
            test_meta
            .groupby("type", group_keys=False)
            .sample(n=1, random_state=42)
            .sort_values("type", key=lambda s: s.map(lambda x: (x != "good", x)))
        )
        
        types = test_samples["type"].replace("good", "normal").tolist()
        num_type = len(types)

        fig = make_subplots(
            rows=2,
            cols=num_type,
            subplot_titles=types,
            vertical_spacing=0.02
        )

        projection_traces = {"UMAP": [], "t-SNE": []}
        for col, (idx, record) in enumerate(test_samples.iterrows(), start=1):
            patch_labels = []
            patch_coords = []

            path = str(ROOT / record["path"])
            
            if record["type"] == "good":
                patch_labels = [0] * 256

            elif labels_exist:
                patch_labels = all_labels[idx]
                
            else:
                mask_path = path.replace("test", "ground_truth").replace(".png", "_mask.png")

                mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask_img = cv2.resize(
                    mask_img,
                    (224, 224),
                    interpolation=cv2.INTER_NEAREST
                )

                for y in range(0, 224, patch_size):
                    for x in range(0, 224, patch_size):
                        patch = mask_img[y:y+14, x:x+14]
                        patch_labels.append(int(np.any(patch > 0)))

                        patch_coords.append((x // patch_size, y // patch_size))   
            
            patch_labels = np.array(patch_labels, dtype=bool)
            
            if len(patch_coords) == 0:
                patch_coords = [
                    (x, y)
                    for y in range(16)
                    for x in range(16)
                ]
            
            patch_coords = np.array(patch_coords)

            patches_umap = umap.fit_transform(patches[idx])
            patches_tsne = tsne.fit_transform(patches[idx])

            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            fig.add_trace(go.Image(z=img), row=1, col=col)

            fig.update_xaxes(visible=False, row=1, col=col)
            fig.update_yaxes(visible=False, row=1, col=col)

            projections = {"UMAP": patches_umap, "t-SNE": patches_tsne}
            for proj_name, dis in projections.items(): 
                fig.add_trace(
                    go.Scattergl(
                        x=dis[~patch_labels, 0],
                        y=dis[~patch_labels, 1],
                        mode="markers",
                        marker=dict(size=4, color="blue"),
                        name="Normal",
                        showlegend=(col == 1),
                        legendgroup="Normal",
                        customdata=patch_coords[~patch_labels],
                        hovertemplate=
                        (
                            "Patch X: %{customdata[0]}"
                            "<br>Patch Y: %{customdata[1]}"
                            "<extra></extra>"
                        ),
                        visible=(proj_name=="UMAP")
                    ),
                    row=2,
                    col=col
                )
                projection_traces[proj_name].append(len(fig.data) - 1)

                fig.add_trace(
                    go.Scattergl(
                        x=dis[patch_labels, 0],
                        y=dis[patch_labels, 1],
                        mode="markers",
                        marker=dict(size=4, color="orange"),
                        name="Defects",
                        showlegend=(col == 2),
                        legendgroup="Defects",
                        customdata=patch_coords[patch_labels],
                        hovertemplate=
                        (
                            "Patch X: %{customdata[0]}"
                            "<br>Patch Y: %{customdata[1]}"
                            "<extra></extra>"
                        ),
                        visible=(proj_name=="UMAP")
                    ),
                    row=2,
                    col=col
                )
                projection_traces[proj_name].append(len(fig.data) - 1)

            n_traces = len(fig.data)

            buttons = []

            for proj_name in projection_traces:

                visible = [False] * n_traces

                # Always show images
                for i, trace in enumerate(fig.data):
                    if trace.type == "image":
                        visible[i] = True

                # Show traces for this projection
                for idx in projection_traces[proj_name]:
                    visible[idx] = True

                buttons.append(
                    dict(
                        label=proj_name,
                        method="update",
                        args=[
                            {"visible": visible}
                        ]
                    )
                )

            fig.update_layout(
                width=220 * num_type,
                height=550,
                updatemenus=[
                    dict(
                        buttons=buttons,
                        direction="down",
                        x=0.0,
                        y=1.15
                    )
                ]
            )

        html_path = HTML_CACHE / f"{cat}_plot.html"

        fig.write_html(html_path)
        fig.write_image(PLOT_DIR / f"{cat}_plot.png")

        plot_figs.append(html_path)
        
    return plot_figs

In [ ]:
if not (HTML_CACHE.exists()):
    plot_figs = plot_patches()

else:
    html_files = list(HTML_CACHE.glob("*_plot.html"))

    if len(html_files) != len(categories):
        plot_figs = plot_patches()
    else:
        plot_figs = html_files

fig_dict = dict(zip(categories, plot_figs))

out = widgets.Output()

dropdown= widgets.Dropdown(
    options=list(fig_dict.keys()),
)

def show_plot(change=None):
    with out:
        clear_output(wait=True)
    
        path = fig_dict[dropdown.value]

        display(
            HTML(path.read_text(encoding="utf-8"))
        )

dropdown.observe(show_plot, names="value")

display(dropdown, out)
show_plot()

Dropdown(options=('bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', …

Output()

Within the patch-level anomaly separation we can see 

In [18]:
def compute_patch_metrics(patches, labels, good_data):
    train_centroid, train_intra = good_data

    sil = silhouette_score(patches, labels)

    good_patches = patches[labels == 0]
    defect_patches = patches[labels == 1]

    # Test inter range from its own centroid
    good_inter = np.mean(np.linalg.norm(good_patches - train_centroid, axis=1))
    defect_inter = np.mean(np.linalg.norm(defect_patches - train_centroid, axis=1))

    # How much further/closer defects are from the train centroid
    sep_ratio = defect_inter / good_inter

    good_centroid = good_patches.mean(axis=0)
    defect_centroid = defect_patches.mean(axis=0)

    good_intra = np.mean(np.linalg.norm(good_patches - good_centroid, axis=1))
    defect_intra = np.mean(np.linalg.norm(defect_patches - defect_centroid, axis=1))

    # How similar the test normal clusters are to the train cluster
    intra_ratio = good_intra / train_intra

    db_index = davies_bouldin_score(patches, labels)
    ch_index = calinski_harabasz_score(patches, labels)
    
    return {
        "silhouette": sil,
        "normal_inter":good_inter,
        "defect_inter": defect_inter,
        "separation": sep_ratio,
        "normal_intra": good_intra,
        "train_intra": train_intra,
        "defect_intra": defect_intra,
        "intra_ratio": intra_ratio,
        "davies_bouldin": db_index,
        "cal_har": ch_index
    }


In [19]:
patch_labels_path = CACHE_DIR / "patch_labels.pkl"
labels_exist = patch_labels_path.exists()

rows = []
all_labels = {}

if labels_exist:
    with open(patch_labels_path, "rb") as f:
        all_labels = pickle.load(f)

for cat in categories:
    good_cat_meta = meta[
            (meta["category"] == cat) &
            (meta["split"] == "test") &
            (meta["type"] == "good")
    ]

    defect_cat_meta = meta[
            (meta["category"] == cat) &
            (meta["split"] == "test") &
            (meta["type"] != "good")
    ]

    good_indices = good_cat_meta.index.to_numpy()
    test_good_patches = patches[good_indices].reshape(-1, patches.shape[-1])

    normal_centroid = test_good_patches.mean(axis=0)
    normal_intra = np.mean(np.linalg.norm(test_good_patches - normal_centroid, axis=1))

    good_data = (normal_centroid, normal_intra)
    
    for idx, record in defect_cat_meta.iterrows():
        if labels_exist:
            patch_labels = all_labels[idx]

        else:
            patch_labels = []

            path = str(ROOT / record["path"])

            mask_path = path.replace("/test/", "/ground_truth/").replace(".png", "_mask.png")

            mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask_img = cv2.resize(
                mask_img, 
                (224, 224),
                interpolation=cv2.INTER_NEAREST
            )

            for y in range(0, 224, patch_size):
                for x in range(0, 224, patch_size):
                    patch = mask_img[y:y+14, x:x+14]
                    patch_labels.append(int(np.any(patch > 0)))

            patch_labels = np.array(patch_labels, dtype=bool)
            all_labels[idx] = patch_labels

        patch_stats = compute_patch_metrics(patches[idx], patch_labels, good_data)
        patch_stats["category"] = cat
        patch_stats["type"] = record["type"]
        patch_stats["path"] = record["path"]
        patch_stats["idx"] = idx

        rows.append(patch_stats)

patch_results = pd.DataFrame(rows)
patch_results.to_csv(RESULTS_DIR / "patch_results.csv")

if not labels_exist:
    with open(CACHE_DIR / "patch_labels.pkl", "wb") as f:
        pickle.dump(all_labels, f)
        

In [20]:
patch_results

,silhouette,normal_inter,defect_inter,separation,normal_intra,train_intra,defect_intra,intra_ratio,davies_bouldin,cal_har,category,type,path,idx
0,0.161342,35.276318,39.305363,1.114214,32.907177,34.671150,27.485424,0.949123,1.920859,41.936706,bottle,contamination,data/mvtec_ad/bottle/test/contamination/008.png,3776
1,0.186531,35.082729,42.254002,1.204410,32.873112,34.671150,29.511862,0.948140,1.887842,37.914439,bottle,contamination,data/mvtec_ad/bottle/test/contamination/005.png,3777
2,0.010051,34.485104,33.457859,0.970212,34.013382,34.671150,23.455458,0.981028,2.382719,8.025347,bottle,contamination,data/mvtec_ad/bottle/test/contamination/019.png,3778
3,0.151055,36.346508,40.229267,1.106826,33.663097,34.671150,29.557268,0.970925,2.148763,33.538239,bottle,contamination,data/mvtec_ad/bottle/test/contamination/006.png,3779
4,0.110735,38.768841,43.954964,1.133770,35.906616,34.671150,22.675386,1.035634,1.731447,17.413124,bottle,contamination,data/mvtec_ad/bottle/test/contamination/013.png,3780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,0.107941,33.142082,40.403622,1.219103,31.642986,32.188232,18.893215,0.983061,1.542459,15.075117,zipper,fabric_interior,data/mvtec_ad/zipper/test/fabric_interior/010.png,733
1254,0.073168,33.931705,38.480145,1.134047,31.253956,32.188232,21.447397,0.970975,1.892298,8.122547,zipper,fabric_interior,data/mvtec_ad/zipper/test/fabric_interior/009.png,734
1255,0.080386,32.876694,37.378471,1.136929,30.957909,32.188232,23.231241,0.961777,2.012367,12.459190,zipper,fabric_interior,data/mvtec_ad/zipper/test/fabric_interior/011.png,735
1256,0.076101,33.476231,39.232796,1.171960,32.260868,32.188232,15.799156,1.002257,1.474674,9.707629,zipper,fabric_interior,data/mvtec_ad/zipper/test/fabric_interior/015.png,736


In [21]:
category_summary = (
    patch_results
    .groupby("category")
    [[
        "silhouette",
        "davies_bouldin",
        "cal_har",
        "normal_inter",
        "defect_inter",
        "separation",
        "normal_intra",
        "defect_intra",
        "train_intra",
        "intra_ratio"
    ]].mean().round(4)
)

category_summary.to_csv(RESULTS_DIR / "patch_embed_summary.csv")
category_summary

,silhouette,davies_bouldin,cal_har,normal_inter,defect_inter,separation,normal_intra,defect_intra,train_intra,intra_ratio
category,,,,,,,,,,
bottle,0.1032,2.0161,21.0375,35.567001,38.716499,1.0888,33.850498,24.612600,34.671200,0.9763
cable,0.1234,2.0764,17.4583,36.312000,42.248100,1.1644,34.226299,27.711201,35.771198,0.9568
capsule,0.1103,1.4781,8.0617,37.648899,45.131500,1.1989,37.051399,20.185600,37.423901,0.9900
carpet,0.2666,1.1308,25.0733,28.021299,44.243401,1.5799,24.875700,15.843400,25.285000,0.9838
grid,0.2727,1.1495,22.8300,29.846300,44.965302,1.5073,24.664900,16.260099,27.479300,0.8976
hazelnut,0.1479,1.4992,19.9355,39.184101,47.635399,1.2157,37.647900,23.349199,38.937401,0.9669
leather,0.2748,0.9460,14.5154,30.895599,49.640900,1.6115,26.494101,12.587400,24.461599,1.0831
metal_nut,0.0924,2.0189,31.2216,37.527302,37.485199,0.9997,34.359699,25.824200,36.552898,0.9400
pill,0.0977,1.7748,14.5052,37.969501,42.797100,1.1274,36.554001,23.924200,37.787300,0.9674


The silhouette scores are generally much higher than those seen for the CLS embeddings, with most categories falling between 0.1 and 0.3. This suggests that patch embeddings form more coherent local structures than global image-level embeddings.

Most categories achieve separation ratios above 1, suggesting that defects are generally positioned further from the normal embedding distribution. Leather (1.61) demonstrates the strongest separation, while metal_nut (≈1.00) exhibits almost no difference between normal and defective patch distances. This indicates that defective metal nut patches occupy nearly the same embedding regions as normal patches.

Most categories have an intra-cluster distance value close to 1, suggesting that defective patches are similarly dispersed to normal patches. However, categories such as leather (1.08) show defective patches becoming more diffuse, while wood (0.82) exhibits more compact defective clusters. This suggests that anomalies affect categories differently, with some defects introducing greater variation and others producing consistent visual patterns.

The screw category performs noticeably better at the patch level than at the CLS embedding level. With stats being:
- Silhoutte ≈ 0.125
- Separation ≈ 1.2
- Intra-ratio ≈ 0.97

One possible explanation is that the anomalous regions within screw images typically occupy only a small portion of the image. As a result, the global CLS embedding is dominated by information from the surrounding normal screw structure, causing normal and defective images to appear highly similar in the image-level embedding space. In contrast, patch embeddings focus on local image regions, allowing patches containing anomalies to be represented separately from normal patches. This suggests that local representations are better able to capture the subtle visual differences associated with screw defects.

In [22]:
defect_summary = (
    patch_results
    .groupby(["category", "type"])
    .agg({
        "silhouette": "mean",
        "separation": "mean",
        "normal_inter": "mean",
        "defect_inter": "mean",
        "normal_intra": "mean",
        "defect_intra": "mean",
        "intra_ratio": "mean"
    })
)

defect_summary.to_csv(RESULTS_DIR / "patch_defect_summary.csv")
defect_summary

silhouette  separation  normal_inter  defect_inter  \
category type                                                                  
bottle   broken_large       0.091542    1.028149     36.186291     37.172180   
         broken_small       0.080272    1.093359     35.056831     38.329891   
         contamination      0.138406    1.141863     35.511585     40.592419   
cable    bent_wire          0.174116    1.265183     35.878395     45.385941   
         cable_swap         0.119747    1.146979     35.810986     41.076069   
...                              ...         ...           ...           ...   
zipper   fabric_border      0.150942    1.316616     33.941998     44.658279   
         fabric_interior    0.083127    1.154388     33.065247     38.192165   
         rough              0.085505    1.065164     32.933510     35.072395   
         split_teeth        0.065362    1.054115     33.195724     34.999119   
         squeezed_teeth     0.036097    1.044792     32.978081     34.454731   

                          normal_intra  defect_intra  intra_ratio  
category type                                                      
bottle   broken_large        33.717354     25.251678     0.972490  
         broken_small        34.080906     23.997528     0.982976  
         contamination       33.735893     24.648438     0.973025  
cable    bent_wire           33.994541     26.905748     0.950334  
         cable_swap          33.917068     31.580320     0.948168  
...                                ...           ...          ...  
zipper   fabric_border       31.417570     17.564991     0.976058  
         fabric_interior     31.426163     18.459604     0.976325  
         rough               31.194031     22.014441     0.969113  
         split_teeth         31.048225     17.049026     0.964583  
         squeezed_teeth      31.148335     16.558025     0.967693  

[73 rows x 7 columns]